# 01 — Metric harness + base-rate invariance check
Work unit 1 (dataset-independent). Confirms the corrected metric math and the load-bearing assumption: S / FNR / HCFN / ECE_atk are base-rate-invariant.

## Session bootstrap

In [ ]:
# === SESSION BOOTSTRAP (run first, every session) ===
# A fresh Colab runtime forgets --global git config, so we re-set it here.
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')          # so `import metrics` finds src/metrics.py
!git pull
print('Session ready - identity set, src/ on path, repo pulled.')

## Metric panel on synthetic data

In [ ]:
import importlib, metrics; importlib.reload(metrics)
from metrics import (all_metrics, severity_risk_sweep, bootstrap_ci,
    severity_S, base_rate_invariance_check, _synthetic)

y, p = _synthetic(seed=0)
for k, v in all_metrics(y, p, t=0.5).items():
    print(f'{k:14s}: {v}')

## Bootstrap 95% CI for S

In [ ]:
pt, lo, hi = bootstrap_ci(severity_S, y, p, n_boot=1000, t=0.5)
print(f'S = {pt:.4f}  95% CI [{lo:.4f}, {hi:.4f}]')

## Load-bearing check: invariance under prevalence sweep

In [ ]:
rows, passed = base_rate_invariance_check(y, p, t=0.5)
print(f"{'pi':>8}{'S':>9}{'FNR':>9}{'HCFN_sh':>9}{'ECE_atk':>10}{'ECE_pool':>10}")
for r in rows:
    print(f"{r['pi']:>8.3f}{r['S']:>9.4f}{r['FNR']:>9.4f}"
          f"{r['HCFN_share']:>9.4f}{r['ECE_atk']:>10.4f}{r['ECE_pooled']:>10.4f}")
print('\nINVARIANCE CHECK:', 'PASS' if passed else 'FAIL')
assert passed, 'Invariance broken - investigate before building further.'

## Figure: invariants flat, pooled ECE moves

In [ ]:
import matplotlib.pyplot as plt
pis = [r['pi'] for r in rows]
fig, ax = plt.subplots(figsize=(6,4))
for key, lab in [('S','S (severity)'),('FNR','FNR'),
                 ('ECE_atk','ECE_atk'),('ECE_pooled','ECE pooled')]:
    ax.plot(pis, [r[key] for r in rows], marker='o', label=lab)
ax.set_xscale('log'); ax.invert_xaxis()
ax.set_xlabel('attack prevalence \u03c0 (log)'); ax.set_ylabel('metric value')
ax.set_title('Base-rate invariance: S/FNR/ECE_atk flat, pooled ECE moves')
ax.legend(); plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/invariance_check.png', dpi=150); plt.show()

---
## Commit + push

In [ ]:
!git add -A
!git commit -m "Metric harness + base-rate invariance figure (folder layout)"
!git push
print('Pushed. Confirm GitHub + Drive in sync.')